## Mutation Testing on Mistral LLM

Using the code generation database from the MongoDB, this notebook will run **zero shot, one shot** and **few shot prompts** on a Mistral LLM. Each prompt technique also includes **no mutation, sequential mutated** and **random mutated** programs. In total, 9 experiments are run through this notebook. All logs are stored in csv files automatically for your analysis.

In [1]:
import os
import sys

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from code_generation.code_generation_tester import CodeGenerationTester
from code_generation.prompt_templates.prompt_template import OpenEndedPromptTemplate
from utility.constants import BigCodeBench, HumanEval, LexicalMutations, SyntacticMutations, LogicalMutations, PromptTypes, CodeGeneration

In [4]:
## Declaring Prompt Type Constants
ZERO_SHOT = PromptTypes.ZERO_SHOT
ONE_SHOT = PromptTypes.ONE_SHOT
FEW_SHOT = PromptTypes.FEW_SHOT

## Declaring Mutation Constants
FOR2WHILE = SyntacticMutations.FOR2WHILE
FOR2ENUMERATE = SyntacticMutations.FOR2ENUMERATE

RANDOM_MUTATION = LexicalMutations.RANDOM
SEQUENTIAL_MUTATION = LexicalMutations.SEQUENTIAL
LITERAL_FORMAT = LexicalMutations.LITERAL_FORMAT

BOOLEAN_LITERAL = LogicalMutations.BOOLEAN_LITERAL
DEMORGAN = LogicalMutations.DEMORGAN
COMMUTATIVE_REORDER = LogicalMutations.COMMUTATIVE_REORDER
CONSTANT_UNFOLD = LogicalMutations.CONSTANT_UNFOLD
CONSTANT_UNFOLD_ADD = LogicalMutations.CONSTANT_UNFOLD_ADD
CONSTANT_UNFOLD_MULT = LogicalMutations.CONSTANT_UNFOLD_MULT

## Declaring Benchmark Name Constants
BIGCODEBENCH = BigCodeBench.NAME
HUMANEVAL = HumanEval.NAME

In [5]:
task_set = HUMANEVAL

try:
    llmtester = CodeGenerationTester(f"{task_set}_Code_Generation")
except Exception as e:
    print(f'llmtester could not launch due to the following error: {e}')

In [6]:
num_tests = llmtester.question_database.count_documents({})

In [7]:
valid_mutations = CodeGeneration.MUTATIONS
print("These are the valid mutation names for code generation:")
for idx, mutation in enumerate(valid_mutations):
    print(idx+1, mutation)

# Run your experiments

In [9]:
# %%script false --no-raise-error
mutations = [RANDOM_MUTATION]
prompt_type = ZERO_SHOT
model_name = "mistral-small-2506"

# Forming the results directory
results_dir =os.path.join(proj_dir, f'results/code_generation/{model_name}')
os.makedirs(results_dir, exist_ok=True)

mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"
output_file_path=f"{results_dir}/{task_set}_{prompt_type}_{mutation_str}.csv"

pass_count = llmtester.run_code_generation_test(
    prompt_helper = OpenEndedPromptTemplate().return_appropriate_prompt(prompt_type),
    # num_tests=num_tests,
    num_tests=10,
    mutations = mutations,
    prompt_type= prompt_type,
    output_file_path=output_file_path,
    task_set = task_set,
)

print(fr"Results saved in {output_file_path}")
